# Conditional Statements

**Topic:** `2.1` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Classify a value by its kind and emptiness

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** truthiness, is None, isinstance

**Explanation and reasoning**

The branch order carries the whole design. None must be tested before the falsy check because None is itself falsy and would otherwise be reported as empty. The bool test sits between the two numeric checks because bool inherits from int and would otherwise be reported as a number.

In [ ]:
def classify(value) -> str:
    if value is None:
        return 'missing'
    if not value:
        return 'empty'
    if isinstance(value, bool):
        return 'other'
    if isinstance(value, (int, float)):
        return 'number'
    return 'other'

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

'0' is truthy text, so it reaches the final branch and is 'other'.

**Alternative approaches**

A table of (test, label) pairs keeps the order visible in data.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert classify(0.0) == 'empty'
assert classify(None) == 'missing'
assert classify(True) == 'other'
assert classify([0]) == 'other'

---

## Solution 2 — Assign a battery band

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** elif, comparisons, thresholds

**Explanation and reasoning**

Each threshold appears exactly once, so the boundary between CRITICAL and LOW is defined by the first test alone. Using <= for the critical band and < for the low band means the value 8.0 has one unambiguous home rather than belonging to both or neither.

In [ ]:
CRITICAL_PCT = 8.0
LOW_PCT = 20.0


def band(pct) -> str:
    if isinstance(pct, bool) or not isinstance(pct, (int, float)):
        raise TypeError(f'battery must be a number, got {type(pct).__name__}')
    if pct <= CRITICAL_PCT:
        return 'CRITICAL'
    if pct < LOW_PCT:
        return 'LOW'
    return 'OK'

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A value above 100 is still 'OK'; the function bands, it does not validate range.

**Alternative approaches**

bisect over a sorted band table is faster for many bands.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert band(8.0) == 'CRITICAL'
assert band(19.9) == 'LOW'
assert band(20.0) == 'OK'
assert band(7.9) == 'CRITICAL'

---

## Solution 3 — Test strictly for a positive number

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** isinstance, bool, comparison

**Explanation and reasoning**

The explicit comparison `> 0` is the point of the exercise: truthiness would answer a different question and would return False for -1 and 0 alike. Rejecting bool explicitly is required because bool is a subclass of int and would otherwise pass the numeric check.

In [ ]:
def is_positive(value) -> bool:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f'expected a real number, got {type(value).__name__}')
    return value > 0

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

-0.0 is not greater than zero, so it returns False rather than raising.

**Alternative approaches**

numbers.Real excludes complex but still admits bool.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert is_positive(0.5) is True
assert is_positive(0.0) is False
assert is_positive(-1.0) is False

---

## Solution 4 — Supply the first usable option

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** or, falsy, defaults

**Explanation and reasoning**

The explicit loop is used here rather than a clever fold because it keeps the result assignment visible and returns the *value*, not a bool. Falsy options are skipped without short-circuiting, so the function still examines every candidate.

In [ ]:
def first_of(*options):
    for option in options:
        if option:
            return option
    return None

**Complexity**

Time O(n) in the options; space O(1).

**Edge cases and failure modes**

No arguments leaves the result as None.

**Alternative approaches**

itertools.chain plus filter is shorter but obscures the default.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert first_of('', None, 'rover-01', 'spare') == 'rover-01'
assert first_of() is None
assert first_of(0, False) is None

---

## Solution 5 — Report the sign of a number

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** if/elif/else, comparison

**Explanation and reasoning**

Testing both signs explicitly leaves zero as the fall-through case, which is the only value that satisfies neither. Deriving zero from a falsy check would be shorter and wrong: it would also catch values that are merely falsy, such as an empty string.

In [ ]:
def sign_of(n) -> str:
    if isinstance(n, bool) or not isinstance(n, (int, float)):
        raise TypeError(f'expected a real number, got {type(n).__name__}')
    if n > 0:
        return 'positive'
    if n < 0:
        return 'negative'
    return 'zero'

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

-0.0 satisfies neither test and correctly returns 'zero'.

**Alternative approaches**

math.copysign distinguishes negative zero, which the words cannot.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert sign_of(-2.5) == 'negative'
assert sign_of(0.0) == 'zero'
assert sign_of(1) == 'positive'

---

## Solution 6 — Test a point against a geofence

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** and, chained comparison, tuples

**Explanation and reasoning**

Validating the shape before unpacking means a malformed fence produces a clear ValueError instead of an unpacking error from the interpreter. Two chained comparisons joined by and express the rectangle directly, with inclusive edges as the specification requires.

In [ ]:
def in_geofence(x, y, bounds) -> bool:
    if not isinstance(bounds, (tuple, list)) or len(bounds) != 4:
        raise ValueError(f'bounds must be four numbers, got {bounds!r}')
    x_min, x_max, y_min, y_max = (float(v) for v in bounds)
    if x_min > x_max or y_min > y_max:
        raise ValueError('bounds are inverted')
    return x_min <= x <= x_max and y_min <= y <= y_max

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A point exactly on a corner is inside the fence.

**Alternative approaches**

A dataclass would give the bounds named fields and better errors.

**Tests — run the cell to verify the reference solution**

In [ ]:
FIELD = (0.0, 50.0, 0.0, 30.0)
assert in_geofence(10, 10, FIELD) is True
assert in_geofence(60, 10, FIELD) is False
assert in_geofence(0, 0, FIELD) is True

---

## Solution 7 — Build a safety interlock

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** guard clauses, elif, dicts

**Explanation and reasoning**

Each guard returns immediately, so the most severe condition is always the one reported. Using state.get for battery and an explicit None test is what distinguishes a missing reading from a reading of zero - a falsy check would report a flat battery as missing, or worse, let it through.

In [ ]:
CRITICAL_BATTERY_PCT = 8.0
LOW_BATTERY_PCT = 20.0
MAX_TILT_DEG = 15.0


def interlock(state: dict) -> tuple[bool, str]:
    if state.get('estop'):
        return False, 'emergency stop is engaged'
    battery = state.get('battery_pct')
    if battery is None:
        return False, 'battery reading is missing'
    if battery <= CRITICAL_BATTERY_PCT:
        return False, 'battery is critical'
    if battery < LOW_BATTERY_PCT:
        return False, 'battery is low'
    if state.get('tilt_deg', 0.0) > MAX_TILT_DEG:
        return False, 'robot is tilted beyond the limit'
    if not state.get('link_ok', True):
        return False, 'radio link is down'
    return True, 'all checks passed'

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

An empty state dict fails on the missing battery reading, not on a crash.

**Alternative approaches**

A list of (test, message) pairs makes the ordering data rather than code.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert interlock({'estop': True, 'battery_pct': 90}) == (False, 'emergency stop is engaged')
assert interlock({'battery_pct': 90}) == (True, 'all checks passed')
assert interlock({})[0] is False

---

## Solution 8 — Grade a score against bands

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dictionaries, iteration, validation

**Explanation and reasoning**

The bands are data, so adding a grade means adding a tuple rather than editing control flow - which is what makes the policy testable. Iterating in descending order and returning on the first match gives the usual 'highest band reached' semantics without any comparison logic.

In [ ]:
def grade(score, bands) -> str:
    if isinstance(score, bool) or not isinstance(score, (int, float)):
        raise TypeError('score must be a number')
    if not 0 <= score <= 100:
        raise ValueError(f'score out of range: {score!r}')
    for minimum, label in bands:
        if score >= minimum:
            return label
    raise ValueError(f'score {score!r} is below every band')

**Complexity**

Time O(b) for b bands; space O(1).

**Edge cases and failure modes**

A score of exactly a band minimum receives that band, because the test is >=.

**Alternative approaches**

bisect_right over a sorted list of minimums is logarithmic.

**Tests — run the cell to verify the reference solution**

In [ ]:
BANDS = ((85, 'A'), (70, 'B'), (50, 'C'))
assert grade(85, BANDS) == 'A'
assert grade(70, BANDS) == 'B'
assert grade(50, BANDS) == 'C'

---

## Solution 9 — Report every failed check at once

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** comprehensions, filtering, aggregation

**Explanation and reasoning**

A single comprehension collects every failure, which is the whole point: a guard chain would report only the first problem and force the operator through several repair cycles. Sorting makes the output deterministic so it can be compared in a test or sent as a stable log field.

In [ ]:
def failed_checks(checks) -> list[str]:
    failures = [label for label, passed in checks if not passed]
    return sorted(failures)

**Complexity**

Time O(n log n) in the checks; space O(f) for failures.

**Edge cases and failure modes**

An empty input produces an empty list rather than raising.

**Alternative approaches**

Return a count as well when only the volume matters.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert failed_checks([('battery', False), ('link', True), ('tilt', False)]) == ['battery', 'tilt']
assert failed_checks([]) == []
assert failed_checks([('a', True)]) == []

---

## Solution 10 — Validate a state-machine transition

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, sets, validation

**Explanation and reasoning**

The get with an empty-set default makes an unknown state a legal no-op rather than a KeyError, which is the right behaviour for a policy check: an unrecognised state is not a permitted move. The rules table is the only source of truth, so the state machine is data a test can inspect.

In [ ]:
def can_transition(current, target, rules: dict) -> bool:
    allowed = rules.get(current, set())
    return target in allowed

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A self-transition is only allowed when the rules list it explicitly.

**Alternative approaches**

Validating the state names and raising is stricter but breaks the fail-safe property.

**Tests — run the cell to verify the reference solution**

In [ ]:
RULES = {'IDLE': {'ACTIVE'}, 'ACTIVE': {'IDLE'}}
assert can_transition('IDLE', 'ACTIVE', RULES) is True
assert can_transition('IDLE', 'STOPPED', RULES) is False
assert can_transition('NOPE', 'IDLE', RULES) is False

---

# Robotics challenge solution — ROBO-X Challenge: Field Robot Drive Interlock

**Explanation**

Building the interlock input from the simulator means the policy is tested against the real values the course already uses, not against invented numbers. The guard chain is inlined so this solution stands alone, and the return shape means the caller can log the reason without parsing text.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

CRITICAL_PCT, LOW_PCT, MAX_TILT_DEG = 8.0, 20.0, 15.0


def interlock(state: dict) -> tuple[bool, str]:
    """Severity-ordered guard chain, self-contained."""
    if state.get('estop'):
        return False, 'emergency stop is engaged'
    battery = state.get('battery_pct')
    if battery is None:
        return False, 'battery reading is missing'
    if battery <= CRITICAL_PCT:
        return False, 'battery is critical'
    if battery < LOW_PCT:
        return False, 'battery is low'
    if state.get('tilt_deg', 0.0) > MAX_TILT_DEG:
        return False, 'robot is tilted beyond the limit'
    return True, 'all checks passed'


def field_state(robot, tilt_deg: float = 2.0) -> dict:
    """Build an interlock input from live telemetry."""
    state = robot.status()
    return {
        'estop': False,
        'battery_pct': state['battery_pct'],
        'tilt_deg': tilt_deg,
    }

**Complexity**

Time O(1); space O(1).

**Notes and trade-offs**

A robot constructed with a tiny battery is rejected as critical.

In [ ]:
robot = SimulatedRobot(name='field-01', battery_wh=48.0)
permitted, reason = interlock(field_state(robot))
assert permitted is True
drained = SimulatedRobot(name='field-02', battery_wh=48.0)
drained.move(distance_m=40.0, speed_mps=1.0)
assert interlock(field_state(drained))[0] is False

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.